# CBB 5740 Biomedical NLP Methods and Application

In this homework, we will implement different prompt engineering techniques and compare their performance using medical Q&A dataset.

The assignment is divided into three tasks:
- Task 1: Develop zero-shot prompt engineering and apply it to the medical dataset.

- Task 2: Develop few-shot prompt engineering (up to to five shots) and apply it to the medical dataset.

- Task 3: Develop advanced prompt engineering (e.g., chain-of-thought, MedPrompt, etc.) and apply it to the medical dataset.


Let's first install all necessary packages.

In [1]:
!pip install openai azure-ai-inference


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
from openai import AzureOpenAI
import json
import random

In this assignment, we use `gpt-5-mini` to perform all experiments. We have set an API Key for everyone that allows you to access to the model through API calls. Before proceeding with the tasks, we need to conduct a simple test to ensure the API key works.

In [3]:
# Input your name (First_Last) and your API key here
# API key removed/renamed to a safe version
my_name = "Andrew_Yu"
api_key = "my_api_key"

In [4]:
# Global configurations
model_name = "gpt-5-mini"
azure_endpoint = "https://ai-zc347-ai-cbb-llm-learning.openai.azure.com/"
api_version = "2024-12-01-preview"

medical_train_data_path = "medicalqa_train_data.jsonl"
medical_test_data_path = "medicalqa_test_data.jsonl"

format_prompt_medical = f"""
Output your answer strictly in the following JSON format:
{{
    "answer": "The letter of the correct option (e.g., A, B, C, D)"
    "explanation": "your explanation here"
}}
"""

In [5]:
# Initialize the AzureOpenAI client
client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_version=api_version,
    api_key=api_key
)

Now, let's run a test. You should see some responses being generated.

In [6]:
response = client.chat.completions.create(
    model=model_name,
    messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "What is the capital of France?"}
    ],
    max_completion_tokens=100
  )

print(response.choices[0].message.content)

The capital of France is Paris.


**Proceed only if the previous cell generated responses.**

## Data Preprocessing

Before proceeding to the tasks, let's examine the data. You should pay attention to the key&value pairs to identify the structure of the dataset. Take one full entry from the dataset to check what the model will be reading.

In [7]:
def load_json(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

In [8]:
medical_train_data = load_json(medical_train_data_path)
medical_train_data[0]

{'question': 'A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?',
 'options': {'A': 'Ampicillin',
  'B': 'Ceftriaxone',
  'C': 'Doxycycline',
  'D': 'Nitrofurantoin'},
 'answer_idx': 'D'}

Take note of the JSON keys and the context format, as they are crucial for developing the prompts in the subsequent tasks. Specifically, the `medical dataset` contains multiple-choice questions.

## Task 1. Develop zero-shot prompt engineering and apply it to the two datasets.

**Zero-shot prompt engineering** involves providing no examples to the large language model and directly requesting an answer.

Using the code framework provided, implement your prompt for each dataset.

In [9]:
# A helper function to format the options for multiple-choice questions
def format_options(options):
    """
    This helper will format the options {"A": "text", "B": "text", ...} into a string format of:
    A: text
    B: text
    ...
    """
    return "\n".join([f"{key}: {value}" for key, value in options.items()])

In [10]:
def build_zero_shot_prompt(data, dataset_type = "medical"):
    # TODO: implement the prompt for zero-shot prompt setting
    # for the medical dataset.

    ############# Your code here ############
    formatted_options = format_options(data['options'])
    prompt = f"""You are a medical expert. Answer the following multiple-choice question by selecting the single best answer. Respond in JSON format only: {{"answer": "X", "explanation": "your reasoning here"}}


Question: {data['question']}
Options:
{formatted_options}
Answer:"""
    
    ########################################

    return prompt


Let's test with a single data from each dataset

In [11]:
print(build_zero_shot_prompt(medical_train_data[0], "medical"))

You are a medical expert. Answer the following multiple-choice question by selecting the single best answer. Respond in JSON format only: {"answer": "X", "explanation": "your reasoning here"}


Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?
Options:
A: Ampicillin
B: Ceftriaxone
C: Doxycycline
D: Nitrofurantoin
Answer:


We will do the final test and evaluation altogether at the end. Now, let's move on to task 2.

## Task 2. Develop few-shot prompt engineering (up to to five shots) and apply it to the two datasets.

**Few-shot Prompt Engineering** involves providing N examples to the large language model prior to presenting the actual test questions.

In [12]:
def format_shot_example(data):
    """
    A helper function to format the example for few-shot prompt setting.
    You may decide to either use or not use this helper function when
    constructing your few-shot prompt.

    This helper will format the json data into a readable string format
    e.g. {'question': '...', 'options': '...', 'answer_idx': '...'} is
    formatted into:
    Options: ...
    question: ...
    Answer: ...
    """
    formatted_options = format_options(data['options'])
    example = f"""
Question: {data['question']}
Options:
{formatted_options}
Answer: {data['answer_idx']}
"""

    return example

In [13]:
def build_few_shot_prompt(data, shots=2):
    # TODO: implement the prompt for few-shot prompt setting.
    # You should implement what the shot example look like and
    # how to combine it with the full prompt.
    # You can use the helper function format_shot_example
    # to format each shot example if you find it helpful.
    assert shots <= 5, "Shots cannot exceed 5."

    shot_examples = random.sample(medical_train_data, shots) # get a shot sample from the dataset

    examples_prompt = ""
    ############# Your code here ############
    for example in shot_examples:
        examples_prompt += format_shot_example(example)

    formatted_options = format_options(data['options'])

    prompt = f"""You are a medical expert. Answer the following multiple-choice question by selecting the single best answer. Respond in JSON format only: {{"answer": "X", "explanation": "your reasoning here"}}


Here are some examples:
{examples_prompt}

Now answer this question:
Question: {data['question']}
Options:
{formatted_options}
Answer:"""
    ########################################

    return prompt

Similarly, let's test if our prompts work.

In [14]:
print(build_few_shot_prompt(medical_train_data[0], shots=2))

You are a medical expert. Answer the following multiple-choice question by selecting the single best answer. Respond in JSON format only: {"answer": "X", "explanation": "your reasoning here"}


Here are some examples:

Question: An 8-year-old boy is brought in by his mother due to complaints of a headache with diminished vision of his temporal field. It has been previously recorded that the patient has poor growth velocity. On imaging, a cystic calcified mass is noted above the sella turcica. From which of the following is this mass most likely derived?
Options:
A: Oral ectoderm
B: Neuroectoderm
C: Neurohypophysis
D: Paraxial mesoderm
Answer: A

Question: A 5-year-old girl with an aortic stenosis correction comes to the office for a follow-up visit for acute lymphoblastic lymphoma. She initiated chemotherapy a week before through a peripherally inserted central line. She reports being ‘tired all the time’ and has been bruising easily. Her vital signs are within normal limits. Physical 

## Task 3: Develop advanced prompt engineering and apply it to the two datasets.

Now, it's your turn to implement some advanced prompt engineering. Some common choices include chain-of-thought or MedPrompt.

Please finish the implementation of the following scratch function. You may implement any helpers as you wish. Document detaily the methods you implemented. And test your output prompt.

In [15]:
def build_cot_example(data):
    """
    This is a helper function to format chain-of-thought examples, prompting the model to give reasoning before the final answer.
    """
    formatted_options = format_options(data['options'])
    example = f"""
Question: {data['question']}
Options:
{formatted_options}
Think through the problem step-by-step:
1. Identify the key clinical findings and context.
2. Consider each option and whether it fits the situation.
3. Eliminate options that are not recommended or less appropriate.
4. Select the best answer.
Answer: {data['answer_idx']}"""
    return example

def build_advanced_prompt(data, shots = 3):
    # TODO: implement the prompt for advanced prompt setting.
    # You can choose to implement any advanced prompting method
    # you like, such as chain-of-thought or MedPrompt.
    # You should implement the full prompt in this function,
    # and you can also implement any helper functions if you
    # find them useful.

    # Method chosen: few-shot chain-of-thought
    # Implementation details: 
    # 1. A small number of examples are added to demonstrate the step-by-step reasoning, and is then followed by the answer to the example
    # 2. Examples are randomly sampled from medical_train_data
    # 3. The test question follows the same format as the examples, but without the final answer, prompting the model to complete the reasoning and answer
    # 4. Included instructions to end the response with "Therefore, the answer is X" so that it is easy to find the final answer at the end of each response
    

    ############# Your code here ############
    assert shots <= 5, "Shots cannot exceed 5."
    shot_examples = random.sample(medical_train_data, shots) # get a shot sample from the dataset
    examples_prompt = ""
    for example in shot_examples:
        examples_prompt += build_cot_example(example)
        examples_prompt += "\n---\n\n"
    formatted_options = format_options(data['options'])
    prompt = f"""You are a medical expert. For each multiple-choice question, reason through the problem step by step before selecting the single best answer. Respond in JSON format only: {{"answer": "X", "explanation": "your reasoning here"}}


Here are some examples:
{examples_prompt}
Now answer this question using the same step-by-step reasoning:
Question: {data['question']}
Options:
{formatted_options}
Think through the problem step-by-step:
1. Identify the key clinical findings and context.
2. Consider each option and whether it fits the situation.
3. Eliminate options that are contraindicated or less appropriate.
4. Select the best answer.
Therefore, the answer is:"""
    ########################################
    return prompt

In [16]:
print(build_advanced_prompt(medical_train_data[0]))

You are a medical expert. For each multiple-choice question, reason through the problem step by step before selecting the single best answer. Respond in JSON format only: {"answer": "X", "explanation": "your reasoning here"}


Here are some examples:

Question: A 51-year-old woman comes to the physician because of numbness of her legs and toes for 3 months. She has also had fatigue and occasional shortness of breath for the past 5 months. She is a painter. Examination shows pale conjunctivae. Sensation to vibration and position is absent over the lower extremities. She has a broad-based gait. The patient sways when she stands with her feet together and closes her eyes. Which of the following laboratory findings is most likely to be seen in this patient?
Options:
A: Poliovirus RNA in cerebrospinal fluid
B: Oligoclonal bands in cerebrospinal fluid
C: Positive rapid plasma reagin test
D: Elevated methylmalonic acid levels
"
Think through the problem step-by-step:
1. Identify the key clini

## Evaluation

Finally, we have completed the implementation of the prompts using various prompt engineering techniques. We will now evaluate and compare their performance to determine the most effective approach. As a reminder, the test data must be used for the final submission.

In [17]:
medical_test_data = load_json(medical_test_data_path)

print(f"Medical test dataset has {len(medical_test_data)} samples.")

Medical test dataset has 100 samples.


In [18]:
def perform_single_experiment(dataset, prompt_type, shots=2):
    if prompt_type == "zero-shot":
        prompt = build_zero_shot_prompt(dataset)
    elif prompt_type == "few-shot":
        prompt = build_few_shot_prompt(dataset, shots=shots)
    elif prompt_type == "advanced":
        prompt = build_advanced_prompt(dataset)
    else:
        raise ValueError("Invalid prompt type. Must be 'zero-shot', 'few-shot', or 'advanced'.")

    system_msg = "You are an medical expert."
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        # max_completion_tokens=4000
    )

    return response.choices[0].message.content

In [19]:
def format_response_to_json(data, response):
    """
    A helper function to format the raw response from the model into a json format.
    e.g. {"answer": "your answer here", "explanation": "your explanation here"}
    will be formatted into a json format of:
    {
        "question": "the question here",
        "output": {
            "answer": "the answer here",
            "explanation": "the explanation here"
        }
    }
    """
    result = {
        "question": data['question'],
        "output": json.loads(response)
    }
    return result

Let's perform one experiment per each prompt type and each dataset to verify our code is working.

In [20]:
for prompt_type in ["zero-shot", "few-shot", "advanced"]:
    print(f"Testing {prompt_type} prompt on medical dataset...", flush=True)

    result = perform_single_experiment(medical_test_data[0], prompt_type, shots=2)
    formatted_result = format_response_to_json(medical_test_data[0], result)

    print(formatted_result)

Testing zero-shot prompt on medical dataset...
{'question': "A 6-year-old African American boy is referred to the hospital by his family physician for jaundice, normocytic anemia, and severe bone pain. He has a history of several episodes of mild bone pain in the past treated with over the counter analgesics. On physical examination, the child is icteric with nonspecific pain in his hands. His hands are swollen, tender, and warm. There is no chest pain, abdominal pain, fever, or hematuria. A complete metabolic panel and complete blood count with manual differential are performed:\nTotal bilirubin\n8.4 mg/dL\nWBC\n9,800/mm3\nHemoglobin \n6.5 g/dL\nMCV 82.3 fL\nPlatelet count  465,000/mm3\nReticulocyte 7%\nPeripheral blood smear shows multiple clumps of elongated and curved cells and erythrocytes with nuclear remnant. The patient's hemoglobin electrophoresis result is pictured below. What is the most likely cause of his condition?", 'output': {'answer': 'B', 'explanation': 'Clinical pict

**Proceed to the final experiments only if the previous cell works**

In [21]:
def perform_experiments(test_data):
    prompt_types = ["zero-shot", "few-shot", "advanced"]
    results = {prompt_type: [] for prompt_type in prompt_types}

    try:
        for data in test_data:
            for prompt_type in prompt_types:
                output = perform_single_experiment(data, prompt_type, shots=2)
                result = {
                    "question": data['question'],
                    "output": json.loads(output)
                }
                results[prompt_type].append(result)

    except Exception as e:
        print(f"An error occurred during experiments: {e}")

    output_file_map = {
        "medical": f"Output1_{my_name}.json"
    }

    with open(output_file_map["medical"], 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=8)
    return results

In [22]:
perform_experiments(medical_test_data)

{'zero-shot': [{'question': "A 6-year-old African American boy is referred to the hospital by his family physician for jaundice, normocytic anemia, and severe bone pain. He has a history of several episodes of mild bone pain in the past treated with over the counter analgesics. On physical examination, the child is icteric with nonspecific pain in his hands. His hands are swollen, tender, and warm. There is no chest pain, abdominal pain, fever, or hematuria. A complete metabolic panel and complete blood count with manual differential are performed:\nTotal bilirubin\n8.4 mg/dL\nWBC\n9,800/mm3\nHemoglobin \n6.5 g/dL\nMCV 82.3 fL\nPlatelet count  465,000/mm3\nReticulocyte 7%\nPeripheral blood smear shows multiple clumps of elongated and curved cells and erythrocytes with nuclear remnant. The patient's hemoglobin electrophoresis result is pictured below. What is the most likely cause of his condition?",
   'output': {'answer': 'B',
    'explanation': 'Clinical picture (dactylitis/hand swel

**Before you submit your final results,** note that we did not provide you any evaluation code. You are expected to implement the evaluation and do the test on the train data by yourself. The answer key for the test data is not provided. We will evaluate based on the correctness of your code and the performance of your prompts on the test data.

Your should submit the following files:
1. Code – Your code file (e.g., Python scripts for the three tasks). For Task 3, document the methods you implemented.
2.	Output_yourname.json – A JSON file containing the output for medical dataset.

The format for the `Output_yourname.json` should look like

```json
{
    "zero-shot": [
        {
            "question": "...",
            "output": {
                "answer": "...",
                "explanation": "..."
            },
        },
        "..."
    ],
    "few-shot": [
        {
            "question": "...",
            "output": {
                "answer": "...",
                "explanation": "..."
            }
        },
        "..."
    ],
    "advanced": [
        {
            "question": "...",
            "output": {
                "answer": "...",
                "explanation": "..."
            }
        },
        "..."
    ]
}
```

In [26]:
def evaluate_on_train(train_data, sample_size=100):
    prompt_types = ["zero-shot", "few-shot", "advanced"]
    results = {prompt_type: {"correct": 0, "total": 0, "errors": 0} for prompt_type in prompt_types}

    sample = random.sample(train_data, min(sample_size, len(train_data)))
    
    for i, data in enumerate(sample):
        for prompt_type in prompt_types:
            try:
                response = perform_single_experiment(data, prompt_type, shots=2)
                parsed = json.loads(response)
                predicted = parsed.get("answer", "").strip().upper()
                correct = data['answer_idx'].strip().upper()
                
                results[prompt_type]["total"] += 1
                if predicted == correct:
                    results[prompt_type]["correct"] += 1
                    
            except Exception as e:
                results[prompt_type]["errors"] += 1
                print(f"Error on {prompt_type}: {e}")
    
    # Print accuracy summary
    print("\n=== Evaluation Results on Training Data ===")
    for prompt_type in prompt_types:
        r = results[prompt_type]
        accuracy = r["correct"] / r["total"] if r["total"] > 0 else 0
        print(f"{prompt_type:12s} | Accuracy: {accuracy:.2%} ({r['correct']}/{r['total']}) | Errors: {r['errors']}")
    
    return results

# Run evaluation on train data
eval_results = evaluate_on_train(medical_train_data, sample_size=200)
print(eval_results)

Error on advanced: the JSON object must be str, bytes or bytearray, not NoneType

=== Evaluation Results on Training Data ===
zero-shot    | Accuracy: 96.00% (192/200) | Errors: 0
few-shot     | Accuracy: 95.00% (190/200) | Errors: 0
advanced     | Accuracy: 94.97% (189/199) | Errors: 1
{'zero-shot': {'correct': 192, 'total': 200, 'errors': 0}, 'few-shot': {'correct': 190, 'total': 200, 'errors': 0}, 'advanced': {'correct': 189, 'total': 199, 'errors': 1}}
